# KBO 팀 분석

팀별 타선 강도, 득점력 트렌드, 시즌별 순위 변화를 시각화합니다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

try:
    fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
    plt.rcParams['font.family'] = 'NanumGothic'
except:
    try: plt.rcParams['font.family'] = 'Malgun Gothic'
    except: pass
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

RAW_DIR = Path('../data/raw')

def load_all():
    files = sorted(RAW_DIR.glob('kbo_*.csv'))
    if not files:
        raise FileNotFoundError('데이터 없음. src/crawl.py 먼저 실행하세요.')
    frames = []
    for f in files:
        d = pd.read_csv(f, dtype=str)
        d['Name'] = d['Name'].str.replace(r'[\u0e00-\u9fff\uac00-\ud7af]+', '', regex=True).str.strip()
        frames.append(d)
    df = pd.concat(frames, ignore_index=True)
    skip = {'Name', 'Team', 'PlayerName', 'KName'}
    num_cols = [c for c in df.columns if c not in skip]
    df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')
    df = df[df['PA'] >= 50].copy()
    return df

df = load_all()
print(f'{len(df)}행 로드  |  시즌: {df["Season"].min():.0f}~{df["Season"].max():.0f}')
print('팀 목록:', sorted(df['Team'].dropna().unique()))

In [ ]:
# ── 팀 타선 집계 함수 ─────────────────────────────────────────────────────────

TEAM_METRICS = ['wRC+', 'WAR', 'AVG', 'OBP', 'SLG', 'PA']

def team_season_agg(min_pa=50):
    """시즌×팀 단위로 타선 스탯 집계 (PA가중 평균)."""
    d = df[df['PA'] >= min_pa].copy()
    agg = {}
    for m in TEAM_METRICS:
        if m not in d.columns: continue
        if m == 'PA':
            agg[m] = ('PA', 'sum')
        elif m in ['wRC+', 'WAR']:
            # PA 가중 평균
            d[f'_{m}_w'] = d[m] * d['PA']
            agg[f'{m}_wavg'] = (f'_{m}_w', 'sum')
            agg[f'{m}_pa_sum'] = ('PA', 'sum')
        else:
            d[f'_{m}_w'] = d[m] * d['PA']
            agg[f'{m}_wavg'] = (f'_{m}_w', 'sum')
            agg[f'{m}_pa_sum'] = ('PA', 'sum')

    g = d.groupby(['Season', 'Team']).agg(**{k: v for k, v in agg.items() if isinstance(v, tuple)})

    for m in [x for x in TEAM_METRICS if x != 'PA']:
        wavg_col = f'{m}_wavg'
        pa_col   = f'{m}_pa_sum'
        if wavg_col in g.columns:
            g[m] = g[wavg_col] / g[pa_col]
            g = g.drop(columns=[wavg_col, pa_col], errors='ignore')

    return g.reset_index()

team_df = team_season_agg()
team_df.head()

In [ ]:
# ── 1. 팀별 wRC+ 트렌드 (멀티 라인) ──────────────────────────────────────────

def plot_team_wrc_trend(start_season=2015):
    d = team_df[team_df['Season'] >= start_season].copy()
    teams = sorted(d['Team'].dropna().unique())

    cmap = plt.cm.get_cmap('tab10', len(teams))
    fig, ax = plt.subplots(figsize=(14, 7))

    for i, team in enumerate(teams):
        t = d[d['Team'] == team].sort_values('Season')
        ax.plot(t['Season'], t['wRC+'], marker='o', linewidth=2,
                markersize=5, label=team, color=cmap(i))

    ax.axhline(100, color='black', linestyle='--', alpha=0.4, linewidth=1, label='리그 평균')
    ax.set_title(f'팀별 wRC+ 추이 ({start_season}~)', fontsize=14, fontweight='bold')
    ax.set_xlabel('시즌'); ax.set_ylabel('wRC+ (PA 가중 평균)')
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('../data/team_wrc_trend.png', bbox_inches='tight')
    plt.show()

plot_team_wrc_trend()

In [ ]:
# ── 2. 특정 시즌 팀 타선 순위 비교 (바 차트) ─────────────────────────────────

def plot_team_ranking(season: int, metric: str = 'wRC+'):
    d = team_df[team_df['Season'] == season].dropna(subset=[metric])
    d = d.sort_values(metric, ascending=True)

    colors = ['#F44336' if v == d[metric].max() else '#2196F3' for v in d[metric]]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(d['Team'], d[metric], color=colors, alpha=0.85, edgecolor='white')

    for bar, val in zip(bars, d[metric]):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}', va='center', fontsize=9)

    if metric == 'wRC+':
        ax.axvline(100, color='gray', linestyle='--', alpha=0.6, label='리그 평균(100)')
        ax.legend()

    ax.set_title(f'{season}시즌 팀 타선 순위 — {metric}', fontsize=13, fontweight='bold')
    ax.set_xlabel(metric)
    ax.grid(alpha=0.3, axis='x')
    plt.tight_layout()
    plt.savefig(f'../data/team_rank_{season}_{metric}.png', bbox_inches='tight')
    plt.show()

# 최근 시즌으로 자동 설정
latest = int(team_df['Season'].max())
plot_team_ranking(latest)

In [ ]:
# ── 3. 팀 타선 종합 레이더 (특정 시즌) ───────────────────────────────────────

def plot_team_radar(season: int, teams_to_show=None):
    d = team_df[team_df['Season'] == season].copy()
    if teams_to_show:
        d = d[d['Team'].isin(teams_to_show)]

    radar_cols = ['wRC+', 'AVG', 'OBP', 'SLG', 'WAR']
    radar_cols = [c for c in radar_cols if c in d.columns]
    d = d.dropna(subset=radar_cols)

    # 정규화
    norm = d[radar_cols].copy()
    for c in radar_cols:
        mn, mx = norm[c].min(), norm[c].max()
        norm[c] = (norm[c] - mn) / (mx - mn + 1e-9)

    angles = np.linspace(0, 2*np.pi, len(radar_cols), endpoint=False).tolist()
    angles += angles[:1]

    teams = d['Team'].tolist()
    n = len(teams)
    cols_per_row = 3
    rows = (n + cols_per_row - 1) // cols_per_row

    fig, axes = plt.subplots(rows, cols_per_row,
                              figsize=(5*cols_per_row, 4.5*rows),
                              subplot_kw=dict(polar=True))
    axes = np.array(axes).flatten()
    fig.suptitle(f'{season}시즌 팀 타선 레이더', fontsize=15, fontweight='bold')

    cmap = plt.cm.get_cmap('tab10', n)
    avg_vals = norm.mean()[radar_cols].tolist() + [norm.mean()[radar_cols[0]]]

    for i, (_, row) in enumerate(d.iterrows()):
        ax = axes[i]
        idx = row.name
        vals = norm.loc[idx, radar_cols].tolist() + [norm.loc[idx, radar_cols[0]]]

        ax.plot(angles, avg_vals, color='gray', linewidth=1, linestyle='--')
        ax.fill(angles, avg_vals, alpha=0.05, color='gray')
        ax.plot(angles, vals, color=cmap(i), linewidth=2)
        ax.fill(angles, vals, alpha=0.2, color=cmap(i))
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(radar_cols, fontsize=9)
        ax.set_title(teams[i], fontsize=11, fontweight='bold')

    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.savefig(f'../data/team_radar_{season}.png', bbox_inches='tight')
    plt.show()

plot_team_radar(latest)

In [ ]:
# ── 4. 팀 WAR 연도별 히트맵 ──────────────────────────────────────────────────

def plot_team_heatmap(metric='wRC+', start_season=2015):
    d = team_df[team_df['Season'] >= start_season].pivot(index='Team', columns='Season', values=metric)
    d = d.dropna(how='all')

    fig, ax = plt.subplots(figsize=(14, 7))
    im = ax.imshow(d.values, aspect='auto', cmap='RdYlGn',
                   vmin=d.stack().quantile(0.1), vmax=d.stack().quantile(0.9))

    ax.set_xticks(range(len(d.columns)))
    ax.set_xticklabels(d.columns.astype(int), rotation=45)
    ax.set_yticks(range(len(d.index)))
    ax.set_yticklabels(d.index)

    for i in range(len(d.index)):
        for j in range(len(d.columns)):
            val = d.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=7.5, color='black')

    plt.colorbar(im, ax=ax, label=metric)
    ax.set_title(f'팀별 {metric} 히트맵 ({start_season}~)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'../data/team_heatmap_{metric}.png', bbox_inches='tight')
    plt.show()

plot_team_heatmap('wRC+')